# 🚗 Toronto Traffic Collision Severity Prediction
## Notebook 2 of 5 — Data Cleaning & Feature Engineering

**Author:** Nishi Bhavesh Patel | Student ID: 501356244

---
### What this notebook does:
- Loads the merged dataset from Notebook 1
- Drops invalid GPS rows and duplicates
- Fills ALL missing values
- Creates the severity target variable (Minor / Major / Fatal)
- Engineers new features (season, hour category, flags)
- One-hot encodes categorical columns
- Saves cleaned dataset: `collisions_cleaned.csv`

### Input file:  `collisions_with_weather.csv`
### Output file: `collisions_cleaned.csv`

---

## Step 1 — Import Libraries

In [18]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

MERGED_FILE  = '../MRP - Final Sem/Data/collisions_with_weather.csv'
CLEANED_FILE = '../MRP - Final Sem/Data/collisions_cleaned.csv'

severity_labels = {0: 'Minor', 1: 'Major', 2: 'Fatal'}
print('Libraries loaded!')

Libraries loaded!


---
## Step 2 — Load Merged Dataset

In [20]:
df = pd.read_csv(MERGED_FILE, low_memory=False)
print(f'Loaded: {df.shape}')
df.head(3)

Loaded: (809034, 30)


,_id,OCC_DATE,OCC_MONTH,OCC_DOW,OCC_YEAR,OCC_HOUR,DIVISION,FATALITIES,INJURY_COLLISIONS,FTR_COLLISIONS,...,geometry,datetime,temperature,humidity,dew_point,precipitation,is_rain,is_snow,is_ice,adverse_weather_w
0,1,1.388550e+12,12,Wednesday,2014,23,D52,NaN,NO,NO,...,"{""coordinates"": [[-79.378427745, 43.65041009]]...",2013-12-31 23:00:00,-8.8,81.0,-11.5,0.0,0,0,0,0
1,2,1.388550e+12,12,Wednesday,2014,23,D32,NaN,YES,NO,...,"{""coordinates"": [[5.6843418860808e-14, 5.08888...",2013-12-31 23:00:00,-8.8,81.0,-11.5,0.0,0,0,0,0
2,3,1.388550e+12,12,Wednesday,2014,23,NSA,NaN,YES,NO,...,"{""coordinates"": [[5.6843418860808e-14, 5.08888...",2013-12-31 23:00:00,-8.8,81.0,-11.5,0.0,0,0,0,0


---
## Step 3 — Data Cleaning

Drop invalid rows, fill ALL missing values in the correct order.

In [21]:
# Drop rows with missing GPS
before = len(df)
df = df.dropna(subset=['LAT_WGS84', 'LONG_WGS84'])
print(f'Dropped {before - len(df):,} rows with no GPS')

# Drop exact duplicates
before = len(df)
df = df.drop_duplicates()
print(f'Dropped {before - len(df):,} duplicate rows')

print(f'Shape after cleaning: {df.shape}')

Dropped 131,978 rows with no GPS
Dropped 0 duplicate rows
Shape after cleaning: (677056, 30)


In [22]:
# Fix FATALITIES — NaN means 0 fatalities (not missing)
df['FATALITIES'] = df['FATALITIES'].fillna(0)

# Fill 4 rows with NaN in involvement columns
for col in ['INJURY_COLLISIONS', 'FTR_COLLISIONS', 'PD_COLLISIONS',
            'AUTOMOBILE', 'MOTORCYCLE', 'PASSENGER', 'BICYCLE', 'PEDESTRIAN']:
    if col in df.columns:
        df[col] = df[col].fillna('NO')

# Fill missing categorical values
for col in ['NEIGHBOURHOOD_158', 'DIVISION']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Replace NSA neighbourhood with Unknown
df['NEIGHBOURHOOD_158'] = df['NEIGHBOURHOOD_158'].replace('NSA', 'Unknown')

# Fill missing numeric weather with median
for col in ['temperature', 'humidity', 'dew_point', 'precipitation']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Fill weather flags with 0
for col in ['is_rain', 'is_snow', 'is_ice', 'adverse_weather_w']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Fix OCC_MONTH
df['OCC_MONTH'] = pd.to_numeric(df['OCC_MONTH'], errors='coerce').fillna(0).astype(int)

# Confirm no NaN remaining
total_nan = df.isnull().sum().sum()
print(f'NaN remaining: {total_nan}')
print('✅ All NaN fixed!' if total_nan == 0 else '⚠️ Still has NaN — check above')

NaN remaining: 0
✅ All NaN fixed!


---
## Step 4 — Create Target Variable (Severity)

- **0 = Minor** — property damage only
- **1 = Major** — one or more injuries
- **2 = Fatal** — one or more fatalities

In [23]:
def get_severity(row):
    if pd.notna(row.get('FATALITIES')) and row['FATALITIES'] > 0:
        return 2
    elif str(row.get('INJURY_COLLISIONS', '')).strip().upper() == 'YES':
        return 1
    else:
        return 0

df['severity'] = df.apply(get_severity, axis=1)

print('Severity distribution:')
for k, v in df['severity'].value_counts().sort_index().items():
    print(f'  {severity_labels[k]:8s} ({k}): {v:,} ({v/len(df)*100:.1f}%)')
print()
print('NOTE: Strong class imbalance — SMOTE applied in Notebook 4')

Severity distribution:
  Minor    (0): 580,492 (85.7%)
  Major    (1): 95,928 (14.2%)
  Fatal    (2): 636 (0.1%)

NOTE: Strong class imbalance — SMOTE applied in Notebook 4


---
## Step 5 — Feature Engineering

In [24]:
# Hour category
def hour_category(row):
    dow  = str(row.get('OCC_DOW', '')).strip()
    hour = row.get('OCC_HOUR', 0)
    if dow in ['Saturday', 'Sunday']:
        return 'weekend'
    elif hour in list(range(7, 10)) + list(range(16, 20)):
        return 'peak'
    elif hour >= 22 or hour <= 4:
        return 'late_night'
    else:
        return 'offpeak'

df['hour_category'] = df.apply(hour_category, axis=1)
print('Hour category:')
print(df['hour_category'].value_counts().to_string())

Hour category:
hour_category
late_night    517362
weekend       159694


In [25]:
# Season
def get_season(m):
    try:
        m = int(m)
        if m in [12, 1, 2]:  return 'Winter'
        elif m in [3, 4, 5]: return 'Spring'
        elif m in [6, 7, 8]: return 'Summer'
        else:                 return 'Fall'
    except:
        return 'Unknown'

df['season'] = df['OCC_MONTH'].apply(get_season)
print('Season:')
print(df['season'].value_counts().to_string())

Season:
season
Winter    179589
Fall      174682
Summer    165860
Spring    156925


In [26]:
# Binary flags
df['is_weekend'] = df['OCC_DOW'].apply(
    lambda x: 1 if str(x).strip() in ['Saturday', 'Sunday'] else 0)
df['is_night'] = df['OCC_HOUR'].apply(
    lambda x: 1 if (x >= 21 or x <= 5) else 0)

# Pedestrian / cyclist involvement
if 'PEDESTRIAN' in df.columns:
    df['pedestrian_involved'] = df['PEDESTRIAN'].apply(
        lambda x: 1 if str(x).strip().upper() == 'YES' else 0)
if 'BICYCLE' in df.columns:
    df['cyclist_involved'] = df['BICYCLE'].apply(
        lambda x: 1 if str(x).strip().upper() == 'YES' else 0)

print('Binary flags:')
print(f'  is_weekend:          {df["is_weekend"].sum():,}')
print(f'  is_night:            {df["is_night"].sum():,}')
if 'pedestrian_involved' in df.columns:
    print(f'  pedestrian_involved: {df["pedestrian_involved"].sum():,}')
if 'cyclist_involved' in df.columns:
    print(f'  cyclist_involved:    {df["cyclist_involved"].sum():,}')

Binary flags:
  is_weekend:          159,694
  is_night:            677,056
  pedestrian_involved: 18,585
  cyclist_involved:    11,982


In [27]:
# Neighbourhood frequency encoding
freq_map = df['NEIGHBOURHOOD_158'].value_counts(normalize=True)
df['neighbourhood_freq'] = df['NEIGHBOURHOOD_158'].map(freq_map)
print(f'neighbourhood_freq sample: {df["neighbourhood_freq"].head(3).tolist()}')

neighbourhood_freq sample: [0.009879537290859249, 0.006312624066546932, 0.010762772946403251]


In [28]:
# One-hot encode categorical columns
df = pd.get_dummies(df, columns=['OCC_DOW', 'season', 'hour_category'], drop_first=False)
print(f'Shape after encoding: {df.shape}')

Shape after encoding: (677056, 48)


---
## Step 6 — Final Overview

In [29]:
print('='*55)
print('  CLEANED DATASET OVERVIEW')
print('='*55)
print(f'  Rows:     {len(df):,}')
print(f'  Columns:  {df.shape[1]}')
print(f'\n  Severity distribution:')
for k, v in df['severity'].value_counts().sort_index().items():
    print(f'    {severity_labels[k]:8s}: {v:,} ({v/len(df)*100:.1f}%)')
print(f'\n  Weather flags:')
for col in ['is_rain', 'is_snow', 'is_ice', 'adverse_weather_w']:
    if col in df.columns:
        print(f'    {col:20s}: {df[col].sum():,} ({df[col].mean()*100:.1f}%)')
print(f'\n  OCC_MONTH range: {df["OCC_MONTH"].min()} to {df["OCC_MONTH"].max()}')
print(f'  NaN remaining:   {df.isnull().sum().sum():,}')
print('='*55)

  CLEANED DATASET OVERVIEW
  Rows:     677,056
  Columns:  48

  Severity distribution:
    Minor   : 580,492 (85.7%)
    Major   : 95,928 (14.2%)
    Fatal   : 636 (0.1%)

  Weather flags:
    is_rain             : 39,583 (5.8%)
    is_snow             : 13,275 (2.0%)
    is_ice              : 8,416 (1.2%)
    adverse_weather_w   : 39,583 (5.8%)

  OCC_MONTH range: 1 to 12
  NaN remaining:   0


---
## Step 7 — Save Cleaned Dataset

In [30]:
df.to_csv(CLEANED_FILE, index=False)
print(f'Saved: {CLEANED_FILE}')
print(f'Final shape: {df.shape}')
print('\n✅ Notebook 2 complete! Open Notebook 3 next.')

Saved: ../MRP - Final Sem/Data/collisions_cleaned.csv
Final shape: (677056, 48)

✅ Notebook 2 complete! Open Notebook 3 next.
